# Q3 — Steel Manufacturing LP: Base Case (Part a)

**Course:** Operations Research 1 — Khalifa University  
**Problem:** Multi-period linear programming model to find the least-cost production and capacity expansion plan for a steel manufacturer across two planning periods (2025 and 2035).

## Problem Summary
A steel company uses three production processes with different costs and capacities.  
Demand grows from **150 kt in 2025** to **200 kt in 2035**.  
Goal: minimise total operational and investment costs over both periods.

## Decision Variables
- `x25[p]` — production level (kt) of process p in 2025
- `x35[p]` — production level (kt) of process p in 2035
- `E[p]`   — capacity expansion (kt) of process p between 2025 and 2035

## Objective
Minimise: `∑ op_cost[p]·x25[p]  +  ∑ (op_cost[p]·x35[p] + inv_cost[p]·E[p])`

In [ ]:
import gurobipy as gp
from gurobipy import GRB

## 1. Data

In [ ]:
processes = [1, 2, 3]

# Installed capacity in 2025 (kt)
cap0     = {1: 120, 2: 50,  3: 0}
# Maximum allowable expansion by 2035 (kt)
delta    = {1: 100, 2: 100, 3: 80}
# Operational cost ($/kt)
op_cost  = {1: 300, 2: 400, 3: 200}
# Investment cost for new capacity ($/kt/yr)
inv_cost = {1: 3000, 2: 4000, 3: 5000}

# Environmental use/emission rates (not active in base case — used in Parts b/c/d)
water       = {1: 5,  2: 3, 3: 1}
electricity = {1: 3,  2: 4, 3: 10}
fuel        = {1: 6,  2: 8, 3: 3}
pollutant   = {1: 10, 2: 2, 3: 5}

demand2025 = 150  # kt
demand2035 = 200  # kt

## 2. Build and Solve the Model

In [ ]:
m = gp.Model("Steel_Base_Case")
m.setParam('OutputFlag', 1)  # Set to 0 to suppress Gurobi solver output

# --- Decision variables ---
x25 = m.addVars(processes, name="x2025", lb=0)  # production in 2025
x35 = m.addVars(processes, name="x2035", lb=0)  # production in 2035
E   = m.addVars(processes, name="Expand", lb=0) # capacity expansion

# --- Objective function ---
m.setObjective(
    gp.quicksum(op_cost[p] * x25[p] for p in processes)
    + gp.quicksum(op_cost[p] * x35[p] + inv_cost[p] * E[p] for p in processes),
    GRB.MINIMIZE
)

# --- Constraints ---
cap25_constrs = {}

for p in processes:
    # 2025: production cannot exceed installed capacity
    cap25_constrs[p] = m.addConstr(x25[p] <= cap0[p], name=f"Cap25_p{p}")
    # 2035: production cannot exceed installed + expanded capacity
    m.addConstr(x35[p] <= cap0[p] + E[p], name=f"Cap35_p{p}")
    # Expansion cannot exceed the maximum allowed
    m.addConstr(E[p] <= delta[p], name=f"MaxExp_p{p}")

# Demand must be met in both periods
m.addConstr(x25.sum() >= demand2025, name="Demand25")
m.addConstr(x35.sum() >= demand2035, name="Demand35")

# --- Solve ---
m.optimize()

## 3. Results

In [ ]:
if m.status == GRB.OPTIMAL:
    print(f"{'='*50}")
    print(f"OPTIMAL TOTAL COST: ${m.objVal:,.2f}")
    print(f"{'='*50}")

    print("\n--- 2025 Production Plan ---")
    for p in processes:
        print(f"  Process {p}: {x25[p].x:.2f} kt  (capacity = {cap0[p]} kt)")
        dual = cap25_constrs[p].Pi
        if abs(dual) > 1e-6:
            print(f"    → Binding constraint. Shadow price: ${dual:.2f} per kt")
            print(f"      (Relaxing this capacity by 1 kt would reduce total cost by ${abs(dual):.2f})")

    print("\n--- 2035 Production & Expansion Plan ---")
    for p in processes:
        cap35 = cap0[p] + E[p].x
        print(f"  Process {p}: produce {x35[p].x:.2f} kt  "
              f"| capacity {cap35:.2f} kt  "
              f"| expanded by {E[p].x:.2f} kt")
else:
    print(f"No optimal solution found. Gurobi status code: {m.status}")

## 4. Interpretation

**Optimal total cost: $203,000**

**2025 Plan:**  
Process 1 runs at full capacity (120 kt). Process 2 fills the remaining 30 kt demand. Process 3 is unused (zero initial capacity).

**Shadow prices (2025):**  
- Process 1: −$100/kt → relaxing its capacity by 1 kt saves $100 in total cost  
- Process 3: −$200/kt → if Process 3 had any initial capacity, it would be the cheapest option at $200/kt operating cost

**2035 Plan:**  
Process 1 expands by 30 kt to reach 150 kt capacity and runs at full utilisation.  
Process 2 uses its existing 50 kt capacity with no expansion needed.  
Process 3 remains unused — its low operating cost ($200/kt) is outweighed by its high investment cost ($5,000/kt/yr).

> See `q3_parts_bcd_environmental.ipynb` for the impact of environmental policy constraints on this base case.